# FS1 Final-Shot — 35 Prepare assets and evidence

Research/precompute-only notebook. It materializes pinned Whisper/Qwen assets, audits audio, creates ASR evidence and performs exactly one fail-open repair attempt for each optional plugin. It does not read TEAM-EVAL or GT and does not change production policy.

**Input:** raw AIC dataset and FS1 master preparation freeze. **Internet:** required for one-time pinned downloads. **Output:** `/kaggle/working/triage_eg_fs1_assets_evidence_v01_bundle.zip`.

In [ ]:
import os
from pathlib import Path
REPO_URL = "https://github.com/Irthn1311/AIC2026_TeamPTK_SGU.git"
REPO_REF = "TRIAGEEG"
ANCHOR = "56c2f37df6841af0e7fe858632ccf8554e8ac4e1"
REPO_DIR = Path(os.environ.get("AIC_REPO_DIR", "/kaggle/working/AIC2026_TeamPTK_SGU"))
RAW_INPUT = Path(os.environ.get("AIC_DATA_ROOT", "/kaggle/input/datasets/nadkli/dataset-aic"))
FREEZE_INPUT = Path(os.environ.get("AIC_FS1_FREEZE_ROOT", "/kaggle/input/datasets/irthn1311/fs1-master-preparation-freeze-2026-08-18"))
OUTPUT_ROOT=Path("/kaggle/working/triage_eg_fs1_assets_evidence_v01")
OUTPUT_ZIP=Path("/kaggle/working/triage_eg_fs1_assets_evidence_v01_bundle.zip")
WHISPER_ID="openai/whisper-large-v3-turbo"; WHISPER_REV="41f01f3fe87f28c78e2fbf8b568835947dd65ed9"
QWEN_ID="Qwen/Qwen2.5-VL-3B-Instruct"; QWEN_REV="66285546d2b821cf421d4f5eb2576359d3770cd3"
OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
print({"required_inputs":{"raw_dataset":str(RAW_INPUT),"fs1_master_freeze":str(FREEZE_INPUT)},"internet_required":True,"output_zip":str(OUTPUT_ZIP)})


In [ ]:
import subprocess, sys
if not (REPO_DIR / ".git").is_dir():
    subprocess.run(["git","clone","--branch",REPO_REF,"--single-branch",REPO_URL,str(REPO_DIR)],check=True)
subprocess.run(["git","fetch","origin",REPO_REF],cwd=REPO_DIR,check=True)
subprocess.run(["git","checkout","--detach",ANCHOR],cwd=REPO_DIR,check=True)
HEAD=subprocess.check_output(["git","rev-parse","HEAD"],cwd=REPO_DIR,text=True).strip()
if HEAD != ANCHOR: raise RuntimeError(f"FS1 anchor mismatch: {HEAD}")
sys.path.insert(0,str(REPO_DIR/"src"))
print({"source_ref":REPO_REF,"HEAD":HEAD,"checkout_mode":"DETACHED_PINNED_ANCHOR"})


In [ ]:
import hashlib, json, platform, shutil, time
import torch
profile={"python":platform.python_version(),"torch":torch.__version__,"cuda":torch.version.cuda,"gpu":torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,"vram_bytes":torch.cuda.get_device_properties(0).total_memory if torch.cuda.is_available() else 0,"free_disk_bytes":shutil.disk_usage("/kaggle/working").free}
if not torch.cuda.is_available(): raise RuntimeError("FS1 Notebook 35 requires Kaggle T4")
(OUTPUT_ROOT/"runtime_profile.json").write_text(json.dumps(profile,indent=2)+"\n")
print(profile)


In [ ]:
from huggingface_hub import snapshot_download
def digest(path):
    h=hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda:f.read(8*1024*1024),b""): h.update(block)
    return h.hexdigest()
def materialize(model_id,revision,name):
    root=Path("/kaggle/working/fs1_model_assets")/name
    snapshot_download(model_id,revision=revision,local_dir=root,ignore_patterns=["*.h5","*.msgpack","*.onnx"])
    files=[{"path":p.relative_to(root).as_posix(),"size_bytes":p.stat().st_size,"sha256":digest(p)} for p in sorted(root.rglob("*")) if p.is_file()]
    return {"model_id":model_id,"exact_revision":revision,"runtime_path":str(root),"files":files,"size_bytes":sum(x["size_bytes"] for x in files)}
assets=[materialize(WHISPER_ID,WHISPER_REV,"whisper-large-v3-turbo"),materialize(QWEN_ID,QWEN_REV,"qwen2.5-vl-3b-instruct")]
(OUTPUT_ROOT/"asset_manifest.json").write_text(json.dumps({"assets":assets},indent=2)+"\n")
print([(x["model_id"],x["size_bytes"]) for x in assets])


In [ ]:
# Audit every video; audio absence/failure is recorded and never blocks.
import subprocess
videos=sorted(p for p in RAW_INPUT.rglob("*") if p.suffix.lower() in {".mp4",".avi",".mkv",".mov"})
audio_audit=[]
for video in videos:
    probe=subprocess.run(["ffprobe","-v","error","-select_streams","a","-show_entries","stream=index,codec_name","-of","json",str(video)],capture_output=True,text=True)
    try: streams=json.loads(probe.stdout or "{}").get("streams",[])
    except json.JSONDecodeError: streams=[]
    audio_audit.append({"video_id":video.stem,"video_path":str(video),"has_audio":bool(streams),"probe_returncode":probe.returncode})
(OUTPUT_ROOT/"audio_audit.jsonl").write_text("".join(json.dumps(x,ensure_ascii=False)+"\n" for x in audio_audit))
print({"videos":len(videos),"with_audio":sum(x["has_audio"] for x in audio_audit)})


In [ ]:
# Sequential Whisper preprocessing. Model unloads before any other heavy model.
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline
whisper_root=Path(assets[0]["runtime_path"])
processor=AutoProcessor.from_pretrained(whisper_root,local_files_only=True)
model=AutoModelForSpeechSeq2Seq.from_pretrained(whisper_root,local_files_only=True,torch_dtype=torch.float16,low_cpu_mem_usage=True).to("cuda").eval()
asr=pipeline("automatic-speech-recognition",model=model,tokenizer=processor.tokenizer,feature_extractor=processor.feature_extractor,torch_dtype=torch.float16,device=0,chunk_length_s=30,return_timestamps=True)
transcripts=[]
for item in audio_audit:
    if not item["has_audio"]: continue
    try:
        result=asr(item["video_path"],generate_kwargs={"task":"transcribe"})
        for chunk in result.get("chunks",[]):
            start,end=chunk.get("timestamp",(0,0)); transcripts.append({"video_id":item["video_id"],"start_seconds":start,"end_seconds":end,"start_frame":None,"end_frame":None,"text":" ".join(chunk.get("text","").split()),"raw_text":chunk.get("text",""),"language":result.get("language"),"model_revision":WHISPER_REV,"provenance":{"video_path":item["video_path"]}})
    except Exception as error: transcripts.append({"video_id":item["video_id"],"status":"ASR_FAILED","error":f"{type(error).__name__}: {error}","model_revision":WHISPER_REV})
(OUTPUT_ROOT/"asr_transcripts.jsonl").write_text("".join(json.dumps(x,ensure_ascii=False)+"\n" for x in transcripts))
del asr,model,processor; torch.cuda.empty_cache()


In [ ]:
# One bounded fail-open repair attempt per optional plugin. Detailed smoke is kept in the manifest.
import gc
from dataclasses import asdict, dataclass

@dataclass
class PluginStatus:
    name: str
    enabled: bool
    status: str
    detail: str = ""

class SequentialModelRegistry:
    """Notebook-local fail-open lifecycle; Notebook 35 must run at the frozen anchor."""
    def __init__(self):
        self.active_name = None
        self.active_model = None
    def load(self, name, loader):
        if self.active_model is not None:
            raise RuntimeError("FS1_SIMULTANEOUS_HEAVY_MODEL_LOAD_FORBIDDEN")
        self.active_name = name
        self.active_model = loader()
        return self.active_model
    def unload(self):
        self.active_name = None
        self.active_model = None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    def repair_optional(self, name, attempt):
        try:
            model = self.load(name, attempt)
            return PluginStatus(name, model is not None, "PASS" if model is not None else f"{name.upper()}_DISABLED", "BOUNDED_REPAIR_ATTEMPT_1")
        except Exception as error:
            return PluginStatus(name, False, f"{name.upper()}_DISABLED", f"{type(error).__name__}: {error}")
        finally:
            self.unload()
import numpy as np
from PIL import Image
registry=SequentialModelRegistry(); plugin_status={}
dummy=Image.fromarray(np.zeros((224,224,3),dtype=np.uint8))
def xclip_smoke():
    from transformers import XCLIPModel,XCLIPProcessor
    root=Path(snapshot_download("microsoft/xclip-base-patch32",revision="a2e27a78a2b5d802e894b8a1ef14f3a8ce490963",local_dir="/kaggle/working/fs1_optional/xclip"))
    processor=XCLIPProcessor.from_pretrained(root,local_files_only=True); model=XCLIPModel.from_pretrained(root,local_files_only=True).to("cuda").eval()
    batch=processor(text=["a person moves"],videos=[dummy]*8,return_tensors="pt",padding=True).to("cuda")
    with torch.inference_mode(): output=model(**batch)
    if output.logits_per_video is None or not torch.isfinite(output.logits_per_video).all(): raise RuntimeError("XCLIP_INVALID_LOGITS")
    return model
def dino_smoke():
    from transformers import AutoModelForZeroShotObjectDetection,AutoProcessor
    root=Path(snapshot_download("IDEA-Research/grounding-dino-tiny",revision="a2bb814dd30d776dcf7e30523b00659f4f141c71",local_dir="/kaggle/working/fs1_optional/dino"))
    processor=AutoProcessor.from_pretrained(root,local_files_only=True); model=AutoModelForZeroShotObjectDetection.from_pretrained(root,local_files_only=True,torch_dtype=torch.float32).to("cuda").eval()
    batch=processor(images=dummy,text="person.",return_tensors="pt").to("cuda")
    with torch.inference_mode(): output=model(**batch)
    if not torch.isfinite(output.logits).all(): raise RuntimeError("DINO_INVALID_LOGITS")
    return model
def sam_smoke():
    from transformers import Sam2Model,Sam2Processor
    root=Path(snapshot_download("facebook/sam2.1-hiera-tiny",revision="de431c4043854a71d8101e17995dfe596bf101a5",local_dir="/kaggle/working/fs1_optional/sam21"))
    processor=Sam2Processor.from_pretrained(root,local_files_only=True); model=Sam2Model.from_pretrained(root,local_files_only=True).to("cuda").eval()
    batch=processor(images=dummy,input_points=[[[[112,112]]]],input_labels=[[[1]]],return_tensors="pt").to("cuda")
    with torch.inference_mode(): output=model(**batch)
    if output.pred_masks is None: raise RuntimeError("SAM_INVALID_MASKS")
    return model
def ppocr_smoke():
    from paddleocr import PaddleOCR
    engine=PaddleOCR(lang="vi",ocr_version="PP-OCRv5",device="gpu")
    result=engine.predict(np.asarray(dummy))
    if result is None: raise RuntimeError("PPOCR_INVALID_OUTPUT")
    return engine
attempts={
 "xclip":xclip_smoke,
 "ppocr":ppocr_smoke,
 "object":dino_smoke,
 "sam":sam_smoke,
}
for name,attempt in attempts.items(): plugin_status[name]=asdict(registry.repair_optional(name,attempt))
(OUTPUT_ROOT/"plugin_status.json").write_text(json.dumps(plugin_status,indent=2)+"\n")


In [ ]:
# Deterministic lexical index and evidence manifest.
import re
lexical={}
for row in transcripts:
    if row.get("text"):
        for token in set(re.findall(r"\w+",row["text"].casefold())): lexical.setdefault(token,[]).append({"video_id":row["video_id"],"start_seconds":row["start_seconds"],"end_seconds":row["end_seconds"]})
(OUTPUT_ROOT/"asr_lexical_index.json").write_text(json.dumps(lexical,ensure_ascii=False,sort_keys=True)+"\n")
evidence_manifest={"source_dataset":str(RAW_INPUT),"whisper_revision":WHISPER_REV,"transcript_rows":len(transcripts),"lexical_terms":len(lexical),"plugin_status":plugin_status}
(OUTPUT_ROOT/"evidence_manifest.json").write_text(json.dumps(evidence_manifest,indent=2)+"\n")


In [ ]:
# Deterministic download bundle; large model snapshots are zipped separately for Kaggle datasets.
for name in ("whisper-large-v3-turbo","qwen2.5-vl-3b-instruct"):
    shutil.make_archive(f"/kaggle/working/fs1_{name}_asset","zip",Path("/kaggle/working/fs1_model_assets")/name)
shutil.make_archive(str(OUTPUT_ZIP.with_suffix("")),"zip",OUTPUT_ROOT)
print({"download_zip":str(OUTPUT_ZIP),"whisper_asset_zip":"/kaggle/working/fs1_whisper-large-v3-turbo_asset.zip","qwen_asset_zip":"/kaggle/working/fs1_qwen2.5-vl-3b-instruct_asset.zip"})
